# LangGraph ReAct Agent — Agentic AI Assignment

This notebook implements a **ReAct-style agentic AI agent** using [LangGraph](https://langchain-ai.github.io/langgraph/).

### What makes this "agentic"?
Unlike a plain chatbot that replies once, this agent:
1. **Reasons** about what it needs to do.
2. **Decides on its own** whether to call a tool (calculator, web search).
3. **Observes** the tool's result.
4. **Loops** back into reasoning until it's ready to give a final answer.

This *think → act → observe* cycle is the core definition of an "agent" in agentic AI.

### Graph shape

```
        START
          |
          v
     +----------+
     |  agent   |  <--------------------+
     +----------+                       |
          |                             |
   should_continue?                     |
      /        \                        |
   tools      END                       |
     |                                  |
     +----------------------------------+
     (tool results are fed back to the agent)
```


## 1. Setup

Run the cell below once to install the required libraries.

In [ ]:
!pip install -q langgraph langchain langchain-core langchain-anthropic langchain-openai duckduckgo-search

## 2. Set your API key

Set an API key for whichever LLM you want to use. By default this notebook uses **Claude (Anthropic)**.

To use OpenAI instead, set `LLM_PROVIDER = "openai"` and provide `OPENAI_API_KEY`.

In [ ]:
import os

# --- Fill in your API key here, or set it as an environment variable beforehand ---
os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY", "")  # <-- paste your key here if not already set

# Uncomment these two lines to use OpenAI instead of Claude:
# os.environ["LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_API_KEY"] = "your-openai-key-here"


## 3. Imports

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


## 4. Define the tools

Tools are plain Python functions the agent can decide to call on its own.
Add or remove tools here depending on what your assignment needs.

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression, e.g. '12 * (4 + 3)'."""
    try:
        # NOTE: eval is used here only for a simple assignment demo.
        # In production code, use a safe math parser instead of eval().
        allowed_chars = set("0123456789+-*/(). ")
        if not set(expression) <= allowed_chars:
            return "Error: expression contains invalid characters."
        return str(eval(expression))
    except Exception as e:
        return f"Error evaluating expression: {e}"


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information on a topic."""
    try:
        from duckduckgo_search import DDGS

        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
        if not results:
            return "No results found."
        return "\n\n".join(
            f"{r['title']}: {r['body']} ({r['href']})" for r in results
        )
    except Exception as e:
        return f"Search failed: {e}"


tools = [calculator, web_search]


## 5. Define the state

This is the "memory" that flows through the graph. `add_messages` automatically
appends new messages instead of overwriting old ones.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


## 6. Define the LLM

Swap this out for whichever provider your assignment requires.

In [ ]:
def get_llm():
    provider = os.getenv("LLM_PROVIDER", "anthropic").lower()

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    else:
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

    # Bind the tools so the LLM knows they exist and can request to call them.
    return llm.bind_tools(tools)


llm_with_tools = get_llm()

SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the available tools when you need "
    "up-to-date information or need to do a calculation. Think step by "
    "step, and only give a final answer once you have everything you need."
)


## 7. Define the graph nodes

- **`agent_node`**: the "thinking" step — the LLM reasons and decides whether to call a tool.
- **`tool_node`**: executes whichever tool the LLM asked for.
- **`should_continue`**: routing function that decides whether to loop back into tools or end.

In [ ]:
def agent_node(state: AgentState) -> AgentState:
    """The 'thinking' node: the LLM reasons about the conversation so far
    and decides whether to call a tool or respond directly."""
    messages = state["messages"]

    # Prepend the system prompt only once, at the start of the conversation.
    if not any(getattr(m, "type", "") == "system" for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(tools)


def should_continue(state: AgentState) -> str:
    """Routing function: decide whether to loop back into tool execution
    or end the conversation, based on whether the LLM asked for a tool call."""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END


## 8. Build the graph

In [ ]:
def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)

    graph.set_entry_point("agent")

    # After the agent thinks, either go run a tool, or end.
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})

    # After a tool runs, always go back to the agent to reason about the result.
    graph.add_edge("tools", "agent")

    return graph.compile()


app = build_graph()


### (Optional) Visualize the graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization needs internet access / extra deps:", e)


## 9. Try it out

Run the cell below and change `user_input` to test different queries:
- `"What is 342 * 17?"` → uses the calculator tool
- `"What's the latest news about SpaceX?"` → uses the web_search tool
- `"Hi, how are you?"` → answers directly, no tool needed

In [ ]:
conversation = []

user_input = "What is 125 * 8, and then search the web for who founded LangChain?"

conversation.append(HumanMessage(content=user_input))
result = app.invoke({"messages": conversation})
conversation = result["messages"]

for m in conversation:
    role = m.type
    print(f"[{role}] {m.content}\n")


## 10. (Optional) Interactive chat loop

Run this cell to chat with the agent turn by turn in this notebook.

In [ ]:
conversation = []

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break

    conversation.append(HumanMessage(content=user_input))
    result = app.invoke({"messages": conversation})
    conversation = result["messages"]

    final_answer = conversation[-1]
    print(f"\nAgent: {final_answer.content}\n")


## Extending this for your assignment

- **Add more tools**: write a new function, decorate it with `@tool`, add it to the `tools` list.
- **Multi-agent setup**: create multiple node functions (e.g. a "researcher" and a "writer"),
  each with their own system prompt, and add edges between them in `build_graph()`.
- **Persistent memory across sessions**: LangGraph supports checkpointing (e.g. `MemorySaver`)
  if you need the agent to remember past conversations between runs.
- **Streaming output**: use `app.stream(...)` instead of `app.invoke(...)` to show intermediate
  steps (useful for demoing the reasoning process to an instructor).